In [18]:
import pandas as pd
import random
from faker import Faker

fake = Faker("es_ES")
random.seed(42)

n_comunidades = 120
comunidades = []

for i in range(1, n_comunidades + 1):

    numero_viviendas = random.randint(20, 220)
    anio_construccion = random.randint(1960, 2022)

    # Reglas simples para dar coherencia
    numero_ascensores = max(1, round(numero_viviendas / random.randint(35, 60)))

    piscina = 1 if numero_viviendas > 70 and random.random() < 0.65 else 0
    jardin = 1 if numero_viviendas > 60 and random.random() < 0.55 else 0
    garaje = 1 if numero_viviendas > 40 and random.random() < 0.75 else 0
    conserjeria = 1 if numero_viviendas > 90 and random.random() < 0.60 else 0

    comunidades.append({
        "nombre": f"Comunidad {fake.street_name()}",
        "direccion": fake.street_address(),
        "codigo_postal": f"280{random.randint(1, 55):02d}",
        "municipio": "Madrid",
        "numero_viviendas": numero_viviendas,
        "anio_construccion": anio_construccion,
        "numero_ascensores": numero_ascensores,
        "piscina": piscina,
        "garaje": garaje,
        "jardin": jardin,
        "conserjeria": conserjeria
    })

df_comunidades = pd.DataFrame(comunidades)

df_comunidades.head()

,nombre,direccion,codigo_postal,municipio,numero_viviendas,anio_construccion,numero_ascensores,piscina,garaje,jardin,conserjeria
0,Comunidad Alameda Emperatriz Checa,Acceso Rosalinda Peñas 12 Apt. 23,28048,Madrid,183,1967,5,0,1,1,1
1,Comunidad Avenida Brunilda Franch,Plaza de Melania Zapata 77,28002,Madrid,159,1965,3,1,1,1,1
2,Comunidad Via Juan Antonio Coloma,Pasadizo de Gala Seco 59 Apt. 15,28052,Madrid,163,1972,3,1,1,1,1
3,Comunidad Urbanización de Celia Rincón,Calle Bartolomé Caparrós 211,28011,Madrid,21,2008,1,0,0,0,0
4,Comunidad Urbanización Alonso Esteban,Cañada de Clara Bastida 80 Apt. 24,28025,Madrid,198,1987,4,1,0,1,1


In [19]:
df_comunidades.shape

(120, 11)

In [20]:
df_comunidades.isnull().sum()

nombre               0
direccion            0
codigo_postal        0
municipio            0
numero_viviendas     0
anio_construccion    0
numero_ascensores    0
piscina              0
garaje               0
jardin               0
conserjeria          0
dtype: int64

In [21]:
categorias = [
    "Limpieza",
    "Ascensores",
    "Jardinería",
    "Piscinas",
    "Fontanería",
    "Electricidad",
    "Seguridad",
    "Garajes",
    "Reparaciones generales"
]

In [22]:
prefijos = {
    "Limpieza": [
        "Limpiezas", "Higiene", "Clean", "Servicios de Limpieza"
    ],
    "Ascensores": [
        "Ascensores", "Elevadores", "Movilidad Vertical"
    ],
    "Jardinería": [
        "Jardines", "Áreas Verdes", "Jardinería"
    ],
    "Piscinas": [
        "Piscinas", "Aqua", "Mantenimiento Acuático"
    ],
    "Fontanería": [
        "Fontanería", "Hidro", "Servicios Hidráulicos"
    ],
    "Electricidad": [
        "Electricidad", "Electro", "Instalaciones Eléctricas"
    ],
    "Seguridad": [
        "Seguridad", "Vigilancia", "Protección"
    ],
    "Garajes": [
        "Garajes", "Parking", "Puertas y Garajes"
    ],
    "Reparaciones generales": [
        "Reformas", "Mantenimiento", "Servicios Integrales"
    ]
}

terminos = [
    "Madrid", "Centro", "Norte", "Sur", "Capital",
    "Metropolitana", "Henares", "Castellana",
    "Manzanares", "Sierra"
]

formas_juridicas = ["S.L.", "S.L.", "S.L.", "S.A."]

proveedores = []

for categoria in categorias:

    # Creamos varios competidores de cada categoría
    for i in range(6):

        nombre = (
            f"{random.choice(prefijos[categoria])} "
            f"{random.choice(terminos)} "
            f"{random.choice(formas_juridicas)}"
        )

        proveedores.append({
            "nombre": nombre,
            "categoria": categoria,
            "telefono": f"+34 91 {random.randint(100,999)} {random.randint(10,99)} {random.randint(10,99)}",
            "email": None,
            "activo": 1 if random.random() < 0.9 else 0
        })

df_proveedores = pd.DataFrame(proveedores)

# Evitamos nombres duplicados
df_proveedores = df_proveedores.drop_duplicates(
    subset="nombre"
).reset_index(drop=True)

df_proveedores.head(10)

,nombre,categoria,telefono,email,activo
0,Clean Castellana S.A.,Limpieza,+34 91 551 53 33,None,0
1,Clean Castellana S.L.,Limpieza,+34 91 941 18 61,None,1
2,Limpiezas Norte S.L.,Limpieza,+34 91 930 82 48,None,1
3,Higiene Centro S.A.,Limpieza,+34 91 720 86 89,None,1
4,Servicios de Limpieza Castellana S.A.,Limpieza,+34 91 404 85 64,None,1
5,Limpiezas Sierra S.L.,Limpieza,+34 91 881 36 90,None,1
6,Movilidad Vertical Centro S.L.,Ascensores,+34 91 345 32 80,None,1
7,Ascensores Henares S.A.,Ascensores,+34 91 805 86 70,None,1
8,Ascensores Capital S.L.,Ascensores,+34 91 819 68 19,None,1
9,Elevadores Sierra S.L.,Ascensores,+34 91 535 24 79,None,1


In [23]:
import unicodedata
import re

def crear_email_empresa(nombre):
    
    texto = unicodedata.normalize("NFKD", nombre)
    texto = texto.encode("ascii", "ignore").decode("ascii")
    
    texto = texto.lower()
    
    # Quitamos formas jurídicas
    texto = re.sub(r"\bs\.?l\.?\b|\bs\.?a\.?\b", "", texto)
    
    # Dejamos solo letras y números
    texto = re.sub(r"[^a-z0-9]", "", texto)
    
    return f"info@{texto}.es"


df_proveedores["email"] = (
    df_proveedores["nombre"]
    .apply(crear_email_empresa)
)

df_proveedores.head()

,nombre,categoria,telefono,email,activo
0,Clean Castellana S.A.,Limpieza,+34 91 551 53 33,info@cleancastellana.es,0
1,Clean Castellana S.L.,Limpieza,+34 91 941 18 61,info@cleancastellana.es,1
2,Limpiezas Norte S.L.,Limpieza,+34 91 930 82 48,info@limpiezasnorte.es,1
3,Higiene Centro S.A.,Limpieza,+34 91 720 86 89,info@higienecentro.es,1
4,Servicios de Limpieza Castellana S.A.,Limpieza,+34 91 404 85 64,info@serviciosdelimpiezacastellana.es,1


In [24]:
df_proveedores = df_proveedores.reset_index(drop=True)
df_proveedores["proveedor_id"] = df_proveedores.index + 1

In [25]:
df_comunidades = df_comunidades.reset_index(drop=True)
df_comunidades["comunidad_id"] = df_comunidades.index + 1

In [26]:
from datetime import date, timedelta

contratos = []

def elegir_proveedor(categoria):
    candidatos = df_proveedores[
        (df_proveedores["categoria"] == categoria) &
        (df_proveedores["activo"] == 1)
    ]

    return random.choice(candidatos["proveedor_id"].tolist())


for _, comunidad in df_comunidades.iterrows():

    comunidad_id = comunidad["comunidad_id"]
    viviendas = comunidad["numero_viviendas"]

    servicios = ["Limpieza"]

    if comunidad["numero_ascensores"] > 0:
        servicios.append("Ascensores")

    if comunidad["jardin"] == 1:
        servicios.append("Jardinería")

    if comunidad["piscina"] == 1:
        servicios.append("Piscinas")

    if comunidad["garaje"] == 1:
        servicios.append("Garajes")

    if comunidad["conserjeria"] == 1:
        servicios.append("Seguridad")

    # Algunos servicios adicionales
    if random.random() < 0.75:
        servicios.append("Electricidad")

    if random.random() < 0.75:
        servicios.append("Fontanería")

    if random.random() < 0.55:
        servicios.append("Reparaciones generales")

    for servicio in servicios:

        proveedor_id = elegir_proveedor(servicio)

        # Coste base por tipo de servicio
        costes_base = {
            "Limpieza": 80,
            "Ascensores": 900,
            "Jardinería": 50,
            "Piscinas": 60,
            "Garajes": 35,
            "Seguridad": 120,
            "Electricidad": 20,
            "Fontanería": 20,
            "Reparaciones generales": 25
        }

        if servicio == "Ascensores":
            importe = (
                comunidad["numero_ascensores"]
                * costes_base[servicio]
                * random.uniform(0.85, 1.25)
            )
        else:
            importe = (
                viviendas
                * costes_base[servicio]
                * random.uniform(0.85, 1.25)
            )

        fecha_inicio = date(
            random.randint(2022, 2025),
            random.randint(1, 12),
            random.randint(1, 28)
        )

        fecha_fin = fecha_inicio + timedelta(
            days=random.choice([365, 730, 1095])
        )

        contratos.append({
            "comunidad_id": comunidad_id,
            "proveedor_id": proveedor_id,
            "servicio": servicio,
            "fecha_inicio": fecha_inicio,
            "fecha_fin": fecha_fin,
            "importe_anual": round(importe, 2),
            "estado": "Activo" if fecha_fin >= date.today() else "Finalizado"
        })

df_contratos = pd.DataFrame(contratos)

df_contratos.head()

,comunidad_id,proveedor_id,servicio,fecha_inicio,fecha_fin,importe_anual,estado
0,1,4,Limpieza,2025-02-05,2026-02-05,17153.59,Finalizado
1,1,10,Ascensores,2024-02-26,2026-02-25,5566.09,Finalizado
2,1,12,Jardinería,2022-08-12,2025-08-11,8745.42,Finalizado
3,1,43,Garajes,2024-02-22,2025-02-21,6941.89,Finalizado
4,1,40,Seguridad,2024-10-08,2027-10-08,18885.82,Activo


In [27]:
import pandas as pd
import random

gastos = []

# Meses desde enero de 2025 hasta agosto de 2026
meses = pd.date_range(
    start="2025-01-01",
    end="2026-08-01",
    freq="MS"
)

for _, contrato in df_contratos.iterrows():

    importe_mensual = contrato["importe_anual"] / 12

    for mes in meses:

        # Solo generamos gasto si el contrato estaba vigente ese mes
        fecha_inicio = pd.Timestamp(contrato["fecha_inicio"])
        fecha_fin = pd.Timestamp(contrato["fecha_fin"])

        if fecha_inicio <= mes <= fecha_fin:

            # Pequeña variación mensual
            importe = importe_mensual * random.uniform(0.95, 1.05)

            gastos.append({
                "comunidad_id": contrato["comunidad_id"],
                "proveedor_id": contrato["proveedor_id"],
                "fecha": mes + pd.Timedelta(
                    days=random.randint(0, 20)
                ),
                "categoria": contrato["servicio"],
                "concepto": f"Servicio mensual de {contrato['servicio'].lower()}",
                "importe": round(importe, 2)
            })

df_gastos = pd.DataFrame(gastos)

df_gastos.head()

,comunidad_id,proveedor_id,fecha,categoria,concepto,importe
0,1,4,2025-03-15,Limpieza,Servicio mensual de limpieza,1447.07
1,1,4,2025-04-08,Limpieza,Servicio mensual de limpieza,1364.41
2,1,4,2025-05-12,Limpieza,Servicio mensual de limpieza,1386.54
3,1,4,2025-06-03,Limpieza,Servicio mensual de limpieza,1380.29
4,1,4,2025-07-02,Limpieza,Servicio mensual de limpieza,1374.06


In [28]:
categorias_extra = [
    "Seguro",
    "Suministros",
    "Reparación extraordinaria",
    "Tasas"
]

gastos_extra = []

for _, comunidad in df_comunidades.iterrows():

    # Entre 5 y 15 gastos extraordinarios por comunidad
    for _ in range(random.randint(5, 15)):

        categoria = random.choice(categorias_extra)

        importe = random.uniform(100, 1500)

        # Comunidades grandes pueden tener reparaciones más caras
        if categoria == "Reparación extraordinaria":
            importe *= comunidad["numero_viviendas"] / 50

        gastos_extra.append({
            "comunidad_id": comunidad["comunidad_id"],
            "proveedor_id": None,
            "fecha": fake.date_between(
                start_date=pd.Timestamp("2025-01-01").date(),
                end_date=pd.Timestamp("2026-08-31").date()
            ),
            "categoria": categoria,
            "concepto": categoria,
            "importe": round(importe, 2)
        })

df_gastos_extra = pd.DataFrame(gastos_extra)

In [29]:
df_gastos = pd.concat(
    [df_gastos, df_gastos_extra],
    ignore_index=True
)

df_gastos.head()

,comunidad_id,proveedor_id,fecha,categoria,concepto,importe
0,1,4,2025-03-15 00:00:00,Limpieza,Servicio mensual de limpieza,1447.07
1,1,4,2025-04-08 00:00:00,Limpieza,Servicio mensual de limpieza,1364.41
2,1,4,2025-05-12 00:00:00,Limpieza,Servicio mensual de limpieza,1386.54
3,1,4,2025-06-03 00:00:00,Limpieza,Servicio mensual de limpieza,1380.29
4,1,4,2025-07-02 00:00:00,Limpieza,Servicio mensual de limpieza,1374.06


In [30]:
df_gastos.shape

(7970, 6)

In [31]:
df_gastos["importe"].describe()

count    7970.000000
mean      611.016228
std       639.819562
min        34.380000
25%       203.575000
50%       382.435000
75%       826.255000
max      6237.230000
Name: importe, dtype: float64

In [32]:
from datetime import datetime, timedelta
import random
import pandas as pd

incidencias = []

tipos_incidencia = {
    "Ascensores": ["Avería de ascensor", "Puerta bloqueada", "Ruido en ascensor"],
    "Fontanería": ["Fuga de agua", "Baja presión", "Atasco"],
    "Electricidad": ["Fallo de iluminación", "Corte eléctrico", "Avería eléctrica"],
    "Piscinas": ["Problema de depuración", "Fuga en piscina", "Calidad del agua"],
    "Jardinería": ["Riego averiado", "Vegetación deteriorada", "Poda necesaria"],
    "Garajes": ["Puerta de garaje averiada", "Problema de acceso", "Iluminación de garaje"],
    "Limpieza": ["Zona común sin limpiar", "Problema de residuos", "Limpieza deficiente"],
    "Seguridad": ["Fallo de acceso", "Cámara averiada", "Problema de seguridad"],
    "Reparaciones generales": ["Daño en zona común", "Humedad", "Elemento deteriorado"]
}

for _, comunidad in df_comunidades.iterrows():

    # Más viviendas = mayor número potencial de incidencias
    base = comunidad["numero_viviendas"] / 8

    # Los edificios antiguos tienen algo más de incidencias
    antiguedad = 2026 - comunidad["anio_construccion"]

    factor_antiguedad = 1 + max(0, antiguedad - 20) / 100

    n_incidencias = int(
        base * factor_antiguedad * random.uniform(0.8, 1.2)
    )

    for _ in range(n_incidencias):

        servicios_posibles = [
            "Fontanería",
            "Electricidad",
            "Limpieza",
            "Reparaciones generales"
        ]

        if comunidad["numero_ascensores"] > 0:
            servicios_posibles.append("Ascensores")

        if comunidad["piscina"] == 1:
            servicios_posibles.append("Piscinas")

        if comunidad["jardin"] == 1:
            servicios_posibles.append("Jardinería")

        if comunidad["garaje"] == 1:
            servicios_posibles.append("Garajes")

        if comunidad["conserjeria"] == 1:
            servicios_posibles.append("Seguridad")

        servicio = random.choice(servicios_posibles)

        # Buscamos proveedor contratado para ese servicio
        contrato = df_contratos[
            (df_contratos["comunidad_id"] == comunidad["comunidad_id"]) &
            (df_contratos["servicio"] == servicio)
        ]

        proveedor_id = (
            contrato.iloc[0]["proveedor_id"]
            if not contrato.empty
            else None
        )

        fecha_apertura = fake.date_time_between(
            start_date=datetime(2025, 1, 1),
            end_date=datetime(2026, 8, 31)
        )

        prioridad = random.choices(
            ["Baja", "Media", "Alta", "Urgente"],
            weights=[20, 50, 25, 5]
        )[0]

        # Tiempo de resolución según prioridad
        rangos = {
            "Baja": (2, 12),
            "Media": (1, 7),
            "Alta": (1, 4),
            "Urgente": (0.1, 1)
        }

        minimo, maximo = rangos[prioridad]

        horas_resolucion = random.uniform(
            minimo * 24,
            maximo * 24
        )

        fecha_cierre = fecha_apertura + timedelta(
            hours=horas_resolucion
        )

        coste = {
            "Baja": random.uniform(30, 250),
            "Media": random.uniform(80, 600),
            "Alta": random.uniform(150, 1500),
            "Urgente": random.uniform(300, 3000)
        }[prioridad]

        incidencias.append({
            "comunidad_id": comunidad["comunidad_id"],
            "proveedor_id": proveedor_id,
            "tipo": random.choice(tipos_incidencia[servicio]),
            "descripcion": f"Incidencia relacionada con {servicio.lower()}",
            "fecha_apertura": fecha_apertura,
            "fecha_cierre": fecha_cierre,
            "estado": "Cerrada",
            "prioridad": prioridad,
            "coste": round(coste, 2)
        })

df_incidencias = pd.DataFrame(incidencias)

df_incidencias.head()

,comunidad_id,proveedor_id,tipo,descripcion,fecha_apertura,fecha_cierre,estado,prioridad,coste
0,1,NaN,Fuga de agua,Incidencia relacionada con fontanería,2025-03-01 01:08:13,2025-03-06 02:27:06.650334,Cerrada,Baja,145.68
1,1,4.0,Zona común sin limpiar,Incidencia relacionada con limpieza,2025-10-27 04:22:28,2025-10-30 20:58:48.581105,Cerrada,Media,109.17
2,1,12.0,Riego averiado,Incidencia relacionada con jardinería,2025-08-23 03:34:18,2025-08-24 20:35:44.724941,Cerrada,Alta,332.98
3,1,43.0,Puerta de garaje averiada,Incidencia relacionada con garajes,2025-10-12 21:25:03,2025-10-13 12:25:07.592196,Cerrada,Urgente,381.07
4,1,40.0,Problema de seguridad,Incidencia relacionada con seguridad,2025-05-23 15:21:06,2025-05-26 15:22:52.282252,Cerrada,Media,582.16


In [33]:
mask_abiertas = (
    df_incidencias.sample(frac=0.05, random_state=42).index
)

df_incidencias.loc[mask_abiertas, "estado"] = "Abierta"
df_incidencias.loc[mask_abiertas, "fecha_cierre"] = pd.NaT
df_incidencias.loc[mask_abiertas, "coste"] = None

In [34]:
viviendas = []

for _, comunidad in df_comunidades.iterrows():

    comunidad_id = comunidad["comunidad_id"]
    n_viviendas = comunidad["numero_viviendas"]

    # Número aproximado de portales según tamaño
    n_portales = max(1, round(n_viviendas / random.randint(25, 40)))

    viviendas_por_portal = max(1, round(n_viviendas / n_portales))

    contador = 0

    for portal_num in range(1, n_portales + 1):

        portal = chr(64 + portal_num) if portal_num <= 26 else str(portal_num)

        while contador < n_viviendas and (
            contador < portal_num * viviendas_por_portal
            or portal_num == n_portales
        ):

            planta = random.randint(0, 8)
            puerta = random.choice(["A", "B", "C", "D"])

            viviendas.append({
                "comunidad_id": comunidad_id,
                "portal": portal,
                "planta": str(planta),
                "puerta": puerta,
                "metros_cuadrados": round(
                    random.uniform(45, 160),
                    2
                )
            })

            contador += 1

df_viviendas = pd.DataFrame(viviendas)

df_viviendas.head()

,comunidad_id,portal,planta,puerta,metros_cuadrados
0,1,A,8,A,98.66
1,1,A,0,B,118.32
2,1,A,6,D,90.22
3,1,A,6,C,100.22
4,1,A,8,A,119.76


In [35]:
df_viviendas.shape

(14544, 5)

In [36]:
df_comunidades["numero_viviendas"].sum()

np.int64(14544)

In [37]:
len(df_viviendas)

14544

In [38]:
df_viviendas = df_viviendas.reset_index(drop=True)
df_viviendas["vivienda_id"] = df_viviendas.index + 1

In [39]:
reservas = []

for _, comunidad in df_comunidades.iterrows():

    comunidad_id = comunidad["comunidad_id"]

    espacios = []

    if comunidad["jardin"] == 1:
        espacios.append("Zona de barbacoa")
        espacios = []

    # Comunidades con piscina pueden tener instalaciones deportivas
    if comunidad["piscina"] == 1:
        espacios.append ("Pista de pádel")


    # Comunidades grandes
    if comunidad["numero_viviendas"] > 80:
        espacios.append("Sala comunitaria")


    # Algunas comunidades con jardín disponen de pista de tenis
    if comunidad["jardin"] == 1:
        espacios.append("Pista de tenis")
    # Si no hay espacios reservables, no generamos reservas
    if not espacios:
        continue

    viviendas_comunidad = df_viviendas[
        df_viviendas["comunidad_id"] == comunidad_id
    ]

    # Número de reservas en función del tamaño de la comunidad
    n_reservas = random.randint(
        max(10, int(comunidad["numero_viviendas"] * 0.5)),
        max(20, int(comunidad["numero_viviendas"] * 2))
    )

    for _ in range(n_reservas):

        vivienda = viviendas_comunidad.sample(1).iloc[0]

        espacio = random.choice(espacios)

        fecha = fake.date_between(
            start_date=pd.Timestamp("2025-01-01").date(),
            end_date=pd.Timestamp("2026-08-31").date()
        )

        hora_inicio = random.choice([
            "09:00:00",
            "10:00:00",
            "11:00:00",
            "12:00:00",
            "16:00:00",
            "17:00:00",
            "18:00:00",
            "19:00:00"
        ])

        hora_inicio_dt = pd.to_datetime(hora_inicio)

        hora_fin_dt = hora_inicio_dt + pd.Timedelta(
            hours=random.choice([1, 2])
        )

        hora_fin = hora_fin_dt.strftime("%H:%M:%S")

        estado = random.choices(
            ["Confirmada", "Cancelada"],
            weights=[90, 10]
        )[0]

        reservas.append({
            "vivienda_id": vivienda["vivienda_id"],
            "comunidad_id": comunidad_id,
            "espacio": espacio,
            "fecha_reserva": fecha,
            "hora_inicio": hora_inicio,
            "hora_fin": hora_fin,
            "estado": estado
        })

df_reservas = pd.DataFrame(reservas)

df_reservas.head()

,vivienda_id,comunidad_id,espacio,fecha_reserva,hora_inicio,hora_fin,estado
0,133,1,Pista de tenis,2025-10-31,18:00:00,20:00:00,Confirmada
1,7,1,Sala comunitaria,2026-01-17,17:00:00,19:00:00,Confirmada
2,34,1,Pista de tenis,2026-06-07,12:00:00,14:00:00,Confirmada
3,144,1,Pista de tenis,2026-01-17,16:00:00,17:00:00,Confirmada
4,180,1,Pista de tenis,2026-07-15,09:00:00,11:00:00,Confirmada


In [40]:
df_reservas["espacio"].unique()

<StringArray>
['Pista de tenis', 'Sala comunitaria', 'Pista de pádel']
Length: 3, dtype: str

In [41]:
print("Comunidades:", df_comunidades.shape)
print("Proveedores:", df_proveedores.shape)
print("Viviendas:", df_viviendas.shape)
print("Contratos:", df_contratos.shape)
print("Gastos:", df_gastos.shape)
print("Incidencias:", df_incidencias.shape)
print("Reservas:", df_reservas.shape)

Comunidades: (120, 12)
Proveedores: (52, 6)
Viviendas: (14544, 6)
Contratos: (716, 7)
Gastos: (7970, 6)
Incidencias: (2079, 9)
Reservas: (17638, 7)


In [42]:
dataframes = {
    "comunidades": df_comunidades,
    "proveedores": df_proveedores,
    "viviendas": df_viviendas,
    "contratos": df_contratos,
    "gastos": df_gastos,
    "incidencias": df_incidencias,
    "reservas": df_reservas
}

for nombre, df in dataframes.items():
    print(f"\n--- {nombre.upper()} ---")
    print(df.isnull().sum())


--- COMUNIDADES ---
nombre               0
direccion            0
codigo_postal        0
municipio            0
numero_viviendas     0
anio_construccion    0
numero_ascensores    0
piscina              0
garaje               0
jardin               0
conserjeria          0
comunidad_id         0
dtype: int64

--- PROVEEDORES ---
nombre          0
categoria       0
telefono        0
email           0
activo          0
proveedor_id    0
dtype: int64

--- VIVIENDAS ---
comunidad_id        0
portal              0
planta              0
puerta              0
metros_cuadrados    0
vivienda_id         0
dtype: int64

--- CONTRATOS ---
comunidad_id     0
proveedor_id     0
servicio         0
fecha_inicio     0
fecha_fin        0
importe_anual    0
estado           0
dtype: int64

--- GASTOS ---
comunidad_id       0
proveedor_id    1202
fecha              0
categoria          0
concepto           0
importe            0
dtype: int64

--- INCIDENCIAS ---
comunidad_id        0
proveedor_id      298

In [43]:
for nombre, df in dataframes.items():
    print(nombre, "duplicados:", df.duplicated().sum())

comunidades duplicados: 0
proveedores duplicados: 0
viviendas duplicados: 0
contratos duplicados: 0
gastos duplicados: 0


incidencias duplicados: 0
reservas duplicados: 1


In [44]:
viviendas_invalidas = df_viviendas[
    ~df_viviendas["comunidad_id"].isin(df_comunidades["comunidad_id"])
]

len(viviendas_invalidas)

0

In [45]:
print(
    "Contratos con comunidad inválida:",
    (~df_contratos["comunidad_id"].isin(df_comunidades["comunidad_id"])).sum()
)

print(
    "Contratos con proveedor inválido:",
    (~df_contratos["proveedor_id"].isin(df_proveedores["proveedor_id"])).sum()
)

Contratos con comunidad inválida: 0
Contratos con proveedor inválido: 0


In [46]:
df_reservas["espacio"].value_counts()

espacio
Sala comunitaria    9060
Pista de pádel      5201
Pista de tenis      3377
Name: count, dtype: int64

In [47]:
from sqlalchemy import create_engine
import urllib

server = "myresidential-sql.database.windows.net"
database = "myresidential_db"

params = urllib.parse.quote_plus(
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server=tcp:{server},1433;"
    f"Database={database};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Authentication=ActiveDirectoryInteractive;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

In [49]:
import pyodbc

pyodbc.drivers()

['SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)',
 'ODBC Driver 18 for SQL Server']

In [50]:
import pyodbc
pyodbc.drivers()

['SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)',
 'ODBC Driver 18 for SQL Server']

In [51]:
from sqlalchemy import create_engine
import urllib

server = "myresidential-sql.database.windows.net"
database = "myresidential_db"

params = urllib.parse.quote_plus(
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server=tcp:{server},1433;"
    f"Database={database};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Authentication=ActiveDirectoryInteractive;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

In [53]:
import pandas as pd

In [56]:
from sqlalchemy import create_engine
import urllib

server = "myresidential-sql.database.windows.net"
database = "myresidential_db"
usuario = "diego.vega@bootcamp-upgrade.com"

params = urllib.parse.quote_plus(
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server=tcp:{server},1433;"
    f"Database={database};"
    f"UID={usuario};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Authentication=ActiveDirectoryInteractive;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

In [57]:
%pip install mssql-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [58]:
import mssql_python

conn = mssql_python.connect(
    "Server=myresidential-sql.database.windows.net;"
    "Database=myresidential_db;"
    "Authentication=ActiveDirectoryDefault;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
)

cursor = conn.cursor()
cursor.execute("SELECT TOP 5 * FROM comunidades")

rows = cursor.fetchall()
print(rows)

[]


In [59]:
columnas_comunidades = [
    "comunidad_id",
    "nombre",
    "direccion",
    "codigo_postal",
    "municipio",
    "numero_viviendas",
    "anio_construccion",
    "numero_ascensores",
    "piscina",
    "garaje",
    "jardin",
    "conserjeria"
]

datos = df_comunidades[columnas_comunidades].values.tolist()

cursor.execute("SET IDENTITY_INSERT comunidades ON")

cursor.executemany("""
    INSERT INTO comunidades (
        comunidad_id,
        nombre,
        direccion,
        codigo_postal,
        municipio,
        numero_viviendas,
        anio_construccion,
        numero_ascensores,
        piscina,
        garaje,
        jardin,
        conserjeria
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", datos)

cursor.execute("SET IDENTITY_INSERT comunidades OFF")

conn.commit()

print("Comunidades insertadas correctamente")

Comunidades insertadas correctamente


In [60]:
cursor.execute("SELECT COUNT(*) FROM comunidades")
print(cursor.fetchone())

(120)


In [61]:
columnas_proveedores = [
    "proveedor_id",
    "nombre",
    "categoria",
    "telefono",
    "email",
    "activo"
]

datos = df_proveedores[columnas_proveedores].values.tolist()

cursor.execute("SET IDENTITY_INSERT proveedores ON")

cursor.executemany("""
    INSERT INTO proveedores (
        proveedor_id,
        nombre,
        categoria,
        telefono,
        email,
        activo
    )
    VALUES (?, ?, ?, ?, ?, ?)
""", datos)

cursor.execute("SET IDENTITY_INSERT proveedores OFF")

conn.commit()

print("Proveedores insertados correctamente")

Proveedores insertados correctamente


In [62]:
cursor.execute("SELECT COUNT(*) FROM proveedores")
print(cursor.fetchone())

(52)


In [63]:
df_viviendas.head()

,comunidad_id,portal,planta,puerta,metros_cuadrados,vivienda_id
0,1,A,8,A,98.66,1
1,1,A,0,B,118.32,2
2,1,A,6,D,90.22,3
3,1,A,6,C,100.22,4
4,1,A,8,A,119.76,5


In [64]:
columnas_viviendas = [
    "vivienda_id",
    "comunidad_id",
    "portal",
    "planta",
    "puerta",
    "metros_cuadrados"
]

datos = df_viviendas[columnas_viviendas].values.tolist()

cursor.execute("SET IDENTITY_INSERT viviendas ON")

cursor.executemany("""
    INSERT INTO viviendas (
        vivienda_id,
        comunidad_id,
        portal,
        planta,
        puerta,
        metros_cuadrados
    )
    VALUES (?, ?, ?, ?, ?, ?)
""", datos)

cursor.execute("SET IDENTITY_INSERT viviendas OFF")

conn.commit()

print("Viviendas insertadas correctamente")

Viviendas insertadas correctamente


In [65]:
cursor.execute("SELECT COUNT(*) FROM viviendas")
print(cursor.fetchone())

(14544)


In [66]:
print(len(df_viviendas))

14544


In [67]:
df_contratos.head()

,comunidad_id,proveedor_id,servicio,fecha_inicio,fecha_fin,importe_anual,estado
0,1,4,Limpieza,2025-02-05,2026-02-05,17153.59,Finalizado
1,1,10,Ascensores,2024-02-26,2026-02-25,5566.09,Finalizado
2,1,12,Jardinería,2022-08-12,2025-08-11,8745.42,Finalizado
3,1,43,Garajes,2024-02-22,2025-02-21,6941.89,Finalizado
4,1,40,Seguridad,2024-10-08,2027-10-08,18885.82,Activo


In [68]:
columnas_contratos = [
    "comunidad_id",
    "proveedor_id",
    "servicio",
    "fecha_inicio",
    "fecha_fin",
    "importe_anual",
    "estado"
]

datos = df_contratos[columnas_contratos].values.tolist()

cursor.executemany("""
    INSERT INTO contratos (
        comunidad_id,
        proveedor_id,
        servicio,
        fecha_inicio,
        fecha_fin,
        importe_anual,
        estado
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", datos)

conn.commit()

print("Contratos insertados correctamente")

Contratos insertados correctamente


In [69]:
cursor.execute("SELECT COUNT(*) FROM contratos")
print(cursor.fetchone())

(716)


In [70]:
print(len(df_contratos))

716


In [71]:
df_gastos.head()

,comunidad_id,proveedor_id,fecha,categoria,concepto,importe
0,1,4,2025-03-15 00:00:00,Limpieza,Servicio mensual de limpieza,1447.07
1,1,4,2025-04-08 00:00:00,Limpieza,Servicio mensual de limpieza,1364.41
2,1,4,2025-05-12 00:00:00,Limpieza,Servicio mensual de limpieza,1386.54
3,1,4,2025-06-03 00:00:00,Limpieza,Servicio mensual de limpieza,1380.29
4,1,4,2025-07-02 00:00:00,Limpieza,Servicio mensual de limpieza,1374.06


In [77]:
columnas_gastos = [
    "comunidad_id",
    "proveedor_id",
    "fecha",
    "categoria",
    "concepto",
    "importe"
]

df_gastos_sql = df_gastos[columnas_gastos].copy()

# Convertimos proveedor_id nulo a None
df_gastos_sql["proveedor_id"] = df_gastos_sql["proveedor_id"].apply(
    lambda x: None if pd.isna(x) else int(x)
)

# Convertimos fecha a datetime
df_gastos_sql["fecha"] = pd.to_datetime(
    df_gastos_sql["fecha"]
)

# Convertimos otros tipos de numpy/pandas a tipos normales de Python
df_gastos_sql["comunidad_id"] = df_gastos_sql["comunidad_id"].astype(int)
df_gastos_sql["importe"] = df_gastos_sql["importe"].astype(float)

datos = list(
    df_gastos_sql.itertuples(index=False, name=None)
)

In [79]:
print(type(datos[0][2]))
print(datos[0])

<class 'pandas.Timestamp'>
(1, 4.0, Timestamp('2025-03-15 00:00:00'), 'Limpieza', 'Servicio mensual de limpieza', 1447.07)


In [80]:
cursor.executemany("""
    INSERT INTO gastos (
        comunidad_id,
        proveedor_id,
        fecha,
        categoria,
        concepto,
        importe
    )
    VALUES (?, ?, ?, ?, ?, ?)
""", datos)

conn.commit()

print("Gastos insertados correctamente")

ProgrammingError: Driver Error: Syntax error or access violation; DDBC Error: [Microsoft][SQL Server]The incoming tabular data stream (TDS) remote procedure call (RPC) protocol stream is incorrect. Parameter 2 (""): The supplied value is not a valid instance of data type float. Check the source data for invalid values. An example of an invalid value is data of numeric type with scale greater than precision.

In [81]:
conn.rollback()

In [82]:
datos = []

for _, fila in df_gastos.iterrows():

    proveedor_id = (
        None
        if pd.isna(fila["proveedor_id"])
        else int(fila["proveedor_id"])
    )

    fecha = pd.to_datetime(fila["fecha"]).to_pydatetime()

    datos.append((
        int(fila["comunidad_id"]),
        proveedor_id,
        fecha,
        str(fila["categoria"]),
        str(fila["concepto"]),
        float(fila["importe"])
    ))

In [83]:
cursor.executemany("""
    INSERT INTO gastos (
        comunidad_id,
        proveedor_id,
        fecha,
        categoria,
        concepto,
        importe
    )
    VALUES (?, ?, ?, ?, ?, ?)
""", datos)

conn.commit()

print("Gastos insertados correctamente")

Gastos insertados correctamente


In [84]:
cursor.execute("SELECT COUNT(*) FROM gastos")
print("Azure:", cursor.fetchone()[0])

print("Python:", len(df_gastos))

Azure: 7970
Python: 7970


In [85]:
datos_incidencias = []

for _, fila in df_incidencias.iterrows():

    proveedor_id = (
        None
        if pd.isna(fila["proveedor_id"])
        else int(fila["proveedor_id"])
    )

    fecha_apertura = pd.to_datetime(
        fila["fecha_apertura"]
    ).to_pydatetime()

    fecha_cierre = (
        None
        if pd.isna(fila["fecha_cierre"])
        else pd.to_datetime(fila["fecha_cierre"]).to_pydatetime()
    )

    coste = (
        None
        if pd.isna(fila["coste"])
        else float(fila["coste"])
    )

    datos_incidencias.append((
        int(fila["comunidad_id"]),
        proveedor_id,
        str(fila["tipo"]),
        str(fila["descripcion"]),
        fecha_apertura,
        fecha_cierre,
        str(fila["estado"]),
        str(fila["prioridad"]),
        coste
    ))

In [86]:
cursor.executemany("""
    INSERT INTO incidencias (
        comunidad_id,
        proveedor_id,
        tipo,
        descripcion,
        fecha_apertura,
        fecha_cierre,
        estado,
        prioridad,
        coste
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""", datos_incidencias)

conn.commit()

print("Incidencias insertadas correctamente")

Incidencias insertadas correctamente


In [87]:
cursor.execute("SELECT COUNT(*) FROM incidencias")
print("Azure:", cursor.fetchone()[0])

print("Python:", len(df_incidencias))

Azure: 2079
Python: 2079


In [88]:
datos_reservas = []

for _, fila in df_reservas.iterrows():

    fecha_reserva = pd.to_datetime(
        fila["fecha_reserva"]
    ).to_pydatetime()

    datos_reservas.append((
        int(fila["vivienda_id"]),
        int(fila["comunidad_id"]),
        str(fila["espacio"]),
        fecha_reserva,
        str(fila["hora_inicio"]),
        str(fila["hora_fin"]),
        str(fila["estado"])
    ))

In [89]:
print(datos_reservas[0])

(133, 1, 'Pista de tenis', datetime.datetime(2025, 10, 31, 0, 0), '18:00:00', '20:00:00', 'Confirmada')


In [90]:
cursor.executemany("""
    INSERT INTO reservas (
        vivienda_id,
        comunidad_id,
        espacio,
        fecha_reserva,
        hora_inicio,
        hora_fin,
        estado
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", datos_reservas)

conn.commit()

print("Reservas insertadas correctamente")

Reservas insertadas correctamente


In [91]:
cursor.execute("SELECT COUNT(*) FROM reservas")
print("Azure:", cursor.fetchone()[0])

print("Python:", len(df_reservas))

Azure: 17638
Python: 17638
